In [9]:
# %%
import os
import logging
from datetime import datetime
from pathlib import Path
from typing import Annotated, Literal, TypedDict, List, Dict, Any, Optional
from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits import SQLDatabaseToolkit
from langchain_core.messages import AIMessage, ToolMessage, HumanMessage
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langgraph.graph import END, StateGraph, START
from langgraph.graph.message import add_messages
from langchain_core.runnables import RunnableLambda
from langgraph.prebuilt import ToolNode
from langgraph.errors import GraphRecursionError
from langchain_core.prompts import ChatPromptTemplate

import os
from dotenv import load_dotenv

load_dotenv()

# ======================================================================
# Step 1: Logger Setup (FILE ONLY)
# ======================================================================

def get_logger(name: str) -> logging.Logger:
    """Creates a logger that writes only to file: logger/agent.log (no terminal output)"""
    logger = logging.getLogger(name)
    logger.setLevel(logging.INFO)
    logger.propagate = False  # prevent logs from propagating to root (terminal)
    # Remove any existing handlers (including StreamHandler for terminal)
    logger.handlers = []
    log_dir = Path("logger")
    log_dir.mkdir(exist_ok=True)
    log_file = log_dir / "agent.log"
    handler = logging.FileHandler(log_file, encoding="utf-8")
    formatter = logging.Formatter(
        fmt="[{asctime}] {levelname:8s} | {message}",
        datefmt="%Y-%m-%d %H:%M:%S",
        style="{"
    )
    handler.setFormatter(formatter)
    logger.addHandler(handler)
    return logger


# ======================================================================
# Step 2: SQL Agent (Enhanced Logging Version)
# ======================================================================

class State(TypedDict):
    messages: Annotated[list[HumanMessage | AIMessage | ToolMessage], add_messages]
    query_attempts: int
    final_answer: Optional[str]


class SQLAgent:
    def __init__(self, db_path: str, model_name: str = "llama-3.1-8b-instant", groq_api_key: Optional[str] = None):
        self.logger = get_logger("SQLAgent")
        start_init = datetime.now()

        self.connection_string = f"sqlite:///{db_path}"
        self.db = SQLDatabase.from_uri(self.connection_string)
        self.llm = ChatGroq(
            model=model_name,
            api_key=groq_api_key or os.getenv("GROQ_API_KEY"),
            temperature=0,
        )

        self.logger.info("🚀 Logger initialized. Starting SQLAgent setup...")
        self._setup_tools()
        self._setup_strong_prompts()
        self._build_graph()

        total_init_time = (datetime.now() - start_init).total_seconds()
        self.logger.info(f"✅ SQLAgent initialized successfully in {total_init_time:.2f}s")
        self.logger.info("=" * 80)

    # ------------------------------------------------------------------
    def _setup_tools(self):
        self.logger.info("🔧 Setting up SQL tools...")
        start_time = datetime.now()

        toolkit = SQLDatabaseToolkit(db=self.db, llm=self.llm)
        tools = toolkit.get_tools()
        self.list_tables_tool = next(t for t in tools if t.name == "sql_db_list_tables")
        self.get_schema_tool = next(t for t in tools if t.name == "sql_db_schema")

        @tool
        def db_query_tool(query: str) -> str:
            """Executes SELECT queries only. Blocks DML/DDL."""
            logger = get_logger("SQLAgent")
            logger.info(f"⚙️ Running db_query_tool with query: {query}")
            if not query.strip().upper().startswith("SELECT"):
                return "Error: Only SELECT queries are allowed."
            try:
                result = self.db.run_no_throw(query)
                logger.info(f"📊 Query result: {result}")
                return str(result) if result else "No results found."
            except Exception as e:
                logger.error(f"❌ Database error: {str(e)}")
                return f"Error: {str(e)}"

        self.db_query_tool = db_query_tool
        elapsed = (datetime.now() - start_time).total_seconds()
        self.logger.info(f"✅ Tools ready in {elapsed:.2f}s")

    # ------------------------------------------------------------------
    def _setup_strong_prompts(self):
        start_time = datetime.now()
        self.logger.info("🧠 Setting up prompts...")

        self.query_gen_prompt = ChatPromptTemplate.from_messages([
            ("system", """YOU ARE A STRICT SQL QUERY GENERATOR... (same prompt)"""),
            ("placeholder", "{messages}")
        ])

        self.interpret_prompt = ChatPromptTemplate.from_messages([
            ("system", """YOU ARE A DATA ANALYST... (same prompt)"""),
            ("placeholder", "{messages}")
        ])

        elapsed = (datetime.now() - start_time).total_seconds()
        self.logger.info(f"✅ Prompts ready in {elapsed:.2f}s")

    # ------------------------------------------------------------------
    def _create_tool_node_with_fallback(self, tools: list) -> RunnableLambda:
        def handle_tool_error(state: Dict) -> Dict:
            self.logger.error("⚠️ Tool execution error encountered.")
            error = state.get("error")
            tool_calls = state.get("messages", [])[-1].tool_calls if state.get("messages") else []
            return {
                "messages": [
                    ToolMessage(content=f"Error: {repr(error)}", tool_call_id=tc["id"])
                    for tc in tool_calls
                ]
            }

        return ToolNode(tools).with_fallbacks([RunnableLambda(handle_tool_error)], exception_key="error")

    # ------------------------------------------------------------------
    def _build_graph(self):
        self.logger.info("⚙️ Building workflow graph...")
        start_time = datetime.now()

        workflow = StateGraph(State)

        def first_tool_call(state: State) -> Dict:
            self.logger.info("🧩 Node: first_tool_call")
            return {
                "messages": [AIMessage(content="", tool_calls=[{
                    "name": "sql_db_list_tables",
                    "args": {},
                    "id": "init_001"
                }])],
                "query_attempts": 0,
                "final_answer": None
            }

        def model_get_schema(state: State) -> Dict:
            self.logger.info("🧩 Node: model_get_schema")
            return {"messages": [self.llm.bind_tools([self.get_schema_tool]).invoke(state["messages"])]}

        def query_gen_node(state: State) -> Dict:
            self.logger.info("🧩 Node: query_gen_node (Attempt %d)", state.get("query_attempts", 0) + 1)
            response = (self.query_gen_prompt | self.llm).invoke({"messages": state["messages"]})
            self.logger.info(f"🧾 Generated SQL: {response.content.strip()}")
            return {"messages": [response], "query_attempts": state.get("query_attempts", 0) + 1}

        def execute_query_node(state: State) -> Dict:
            self.logger.info("🧩 Node: execute_query_node")
            start_q = datetime.now()
            sql = state["messages"][-1].content.strip()
            try:
                result = self.db.run_no_throw(sql)
                content = str(result) if result else "No results found."
                elapsed = (datetime.now() - start_q).total_seconds()
                self.logger.info(f"✅ SQL executed in {elapsed:.2f}s")
                self.logger.info(f"📋 Query Result: {content}")
            except Exception as e:
                content = f"Error: {str(e)}"
                self.logger.error(f"❌ SQL execution failed: {str(e)}")
            return {"messages": [ToolMessage(content=content, tool_call_id="exec_001")]}

        def interpret_results_node(state: State) -> Dict:
            self.logger.info("🧩 Node: interpret_results_node")
            interp_start = datetime.now()
            interpretation = (self.interpret_prompt | self.llm).invoke({"messages": state["messages"]})
            elapsed = (datetime.now() - interp_start).total_seconds()
            self.logger.info(f"💬 Interpretation ready in {elapsed:.2f}s")
            self.logger.info(f"🧠 Final Answer: {interpretation.content.strip()}")
            return {"messages": [interpretation], "final_answer": interpretation.content}

        # Graph connections
        workflow.add_node("first_tool_call", first_tool_call)
        workflow.add_node("list_tables_tool", self._create_tool_node_with_fallback([self.list_tables_tool]))
        workflow.add_node("model_get_schema", model_get_schema)
        workflow.add_node("get_schema_tool", self._create_tool_node_with_fallback([self.get_schema_tool]))
        workflow.add_node("query_gen", query_gen_node)
        workflow.add_node("execute_query", execute_query_node)
        workflow.add_node("interpret_results", interpret_results_node)

        workflow.add_edge(START, "first_tool_call")
        workflow.add_edge("first_tool_call", "list_tables_tool")
        workflow.add_edge("list_tables_tool", "model_get_schema")
        workflow.add_edge("model_get_schema", "get_schema_tool")
        workflow.add_edge("get_schema_tool", "query_gen")
        workflow.add_edge("query_gen", "execute_query")
        workflow.add_edge("execute_query", "interpret_results")
        workflow.add_edge("interpret_results", END)

        elapsed = (datetime.now() - start_time).total_seconds()
        self.logger.info(f"✅ Workflow graph built in {elapsed:.2f}s")

        self.app = workflow.compile()

    # ------------------------------------------------------------------
    def query(self, question: str, recursion_limit: int = 10) -> Dict[str, Any]:
        start_time = datetime.now()
        logger = self.logger
        logger.info("=" * 80)
        logger.info(f"📝 New Question: {question}")

        try:
            result = self.app.invoke(
                {"messages": [HumanMessage(content=question)], "query_attempts": 0, "final_answer": None},
                config={"recursion_limit": recursion_limit}
            )

            sql_query = None
            for msg in reversed(result["messages"]):
                if hasattr(msg, "content") and "SELECT" in str(msg.content).upper():
                    sql_query = str(msg.content).strip()
                    break

            answer = result.get("final_answer") or result["messages"][-1].content
            total_time = (datetime.now() - start_time).total_seconds()

            logger.info(f"🧾 SQL Query: {sql_query}")
            logger.info(f"💬 Final Answer: {answer}")
            logger.info(f"⏱️ Total Runtime: {total_time:.2f}s")
            logger.info("=" * 80)

            return {"sql_query": sql_query, "answer": answer}

        except GraphRecursionError:
            logger.error("⚠️ Graph recursion error detected.")
            return {"sql_query": None, "answer": "Query too complex or unclear."}
        except Exception as e:
            logger.error(f"❌ Unexpected Error: {str(e)}")
            return {"sql_query": None, "answer": f"Error: {str(e)}"}


In [10]:
# -------------------------------
# Step 3: Example Usage (Single Question)
# -------------------------------


if __name__ == "__main__":
    # Initialize the SQL Agent with SQLite database
    agent = SQLAgent(
        db_path="university.db",
        model_name="llama-3.1-8b-instant",
        groq_api_key=os.getenv("GROQ_API_KEY"), 
    )
# Example: ask a single question
question = "email of Nabin Gurung"
res = agent.query(question)
print(f" Question: '{question}'")
print(f"Answer: {res['answer']}")


 Question: 'email of Nabin Gurung'
Answer: The email of Nabin Gurung is nabin@stu.uni.edu.
